# LOG05 | 미션 08 | Student Workbook

## 이번 미션에서 하는 일

**AAR을 통한 무인 보급 실패 사례 분석 및 중지 조건 민감도 재설계**

1. 실습 목적 및 개요 역할 및 배경: 학습자는 무인 보급장비 투입 전 상태·책임·중지조건 체크리스트 교육용 실습의 참여자로서 임무를 수행합니다. 핵심 과제: 가상 AAR(사후검토) 일지 속에서 무인 보급 실패 사례의 원인을 사람·절차·정보·환경 의 다면적 영역으로 규명하고, 기존 중지 조건이 지나치게 민감하거나 둔감한 지점을 찾아 재설계합니다. 2. 주의 및 검증 지침 데이터 해석의 엄정성: 자료나 AI 설명을 철저히 검증하지 않으면 미확인 기록이나 모의 회신을 실제 현장 확인이나 승인으로 오인할 수 있으므로 주의해야 합니다. 수행 방식: 본인 판단, AI 초안 및 역할 대화, 근거 대조 및 보완, 최종 기록의 전 과정을 원본 자료 범위 내에서 엄정하게 수행합니다. 3. 산출물 및 소요 시간 안내 결과 파일 목록: reports/log05_redteam_critique_report.md LOG05_M08_forms.md forms/LOG05_M08_forms.md 시간 안내: 원본 예상 시간은 3.5시간이며, CMS 편성은 원본 durationHours 를 올림하여 4시간으로 구성되어 있습니다 (실제 소요 시간은 실습 완료 후 확인).

**학습 목표:** 가상 자료를 읽고 자신의 최초 판단을 적은 다음, AI 초안을 원자료와 대조하여 문서와 검증 근거를 완성합니다. 실제 장비 제어·비행·파괴·무력화 방법을 설계하거나 실행하는 훈련은 아닙니다.

**준비물:** 이 Notebook, 미션 8의 가상 자료 및 빈 작성 양식, 기관이 허용한 AI 환경(가능한 경우).

**학습 환경:** Colab은 인터넷이 필요합니다. 폐쇄망에서는 사전 설치된 Jupyter 또는 별도 Markdown 양식으로 진행할 수 있으며 외부 AI를 사용할 수 없으면 미실시로 기록하세요.

**학습 순서:** 자료 먼저 읽기 → 본인 생각 기록 → AI 질문 → 원문 근거 대조 → 수정 → 저장·제출.

**구분:** 파일 저장 / 실습 수행 / 평가 기준 충족 / 실제 교관 승인은 각각 다른 상태입니다.

## 📦 실습자료 업로드 (모든 미션 공통 · 실습 전 필수)

아래 **「LOG05 실습자료 ZIP 업로드 및 압축 해제」** 코드셀을 **먼저 실행**하세요.

1. 교육기관에서 제공한 **`LOG05_실습프로젝트.zip`**을 준비합니다. 파일명이 다르더라도 LOG05 자료가 담긴 정상적인 ZIP 파일이면 업로드할 수 있습니다.
2. **Google Colab:** 셀 실행 후 나타나는 업로드 창에서 ZIP을 선택하면 자동으로 압축을 해제합니다. **로컬 Jupyter:** ZIP 선택 후 **「선택한 ZIP 압축 해제」** 버튼을 누릅니다.
3. 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

**주의:** 업로드가 끝나지 않았다면 미션 실습을 시작하지 마세요. ZIP을 올렸다는 사실만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.


In [ ]:
# @title 📦 LOG05 실습자료 ZIP 업로드 및 압축 해제 { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

display(Markdown("**실습 시작 전 필수:** 교육기관에서 제공한 `LOG05_실습프로젝트.zip`을 업로드하세요. 이름이 다르더라도 정상적인 `.zip` 파일을 선택할 수 있지만, **LOG05 실습자료가 포함된 ZIP**이어야 합니다. 업로드 및 압축 해제를 완료한 후 다음 실습 단계로 진행하세요."))
try:
    from google.colab import files as _colab_files
    print("📤 업로드 창에서 LOG05 실습자료 ZIP 파일을 선택하세요.")
    _uploaded = _colab_files.upload()
    _zip_files = [(name, content) for name, content in _uploaded.items() if name.lower().endswith('.zip')]
    if not _zip_files:
        print("⚠️ ZIP 파일이 업로드되지 않았습니다. 이 셀을 다시 실행하세요.")
    else:
        if len(_zip_files) > 1:
            print("ℹ️ 여러 ZIP 중 첫 번째 파일을 사용합니다:", _zip_files[0][0])
        _name, _data = _zip_files[0]
        _zip_path = Path.cwd() / Path(_name).name
        _zip_path.write_bytes(_data)
        _log05_extract_zip(_zip_path)
except ImportError:
    import ipywidgets as _widgets
    _file_input = _widgets.FileUpload(accept='.zip', multiple=False, description='ZIP 파일 선택')
    _extract_button = _widgets.Button(description='선택한 ZIP 압축 해제', button_style='success')
    _upload_message = _widgets.Output()
    def _get_uploaded_zip():
        value = _file_input.value
        if not value:
            raise ValueError('먼저 LOG05 실습자료 ZIP을 선택하세요.')
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        name = item['name']
        data = item['content']
        if not name.lower().endswith('.zip'):
            raise ValueError('ZIP 파일만 선택할 수 있습니다.')
        return name, bytes(data)
    def _on_extract(_):
        with _upload_message:
            _upload_message.clear_output()
            try:
                name, data = _get_uploaded_zip()
                target = Path.cwd() / Path(name).name
                target.write_bytes(data)
                _log05_extract_zip(target)
            except Exception as exc:
                print('⚠️ 업로드 또는 압축 해제 실패:', exc)
    _extract_button.on_click(_on_extract)
    display(_widgets.VBox([_file_input, _extract_button, _upload_message]))
    print('로컬 Jupyter: ZIP을 선택한 뒤 「선택한 ZIP 압축 해제」 버튼을 누르세요.')


## 원본 가상 자료와 작성 양식

아래는 첨부된 **LOG05 실습자료 원본**에서 미션별로 추출한 자료입니다. 모의 기록에서 아직 공개되지 않은 카드나 결과는 미리 확정하지 마세요.

미션 8. AAR을 통한 무인 보급 실패 사례 분석 및 중지 조건 민감도 재설계

**가. 제공 자료 1 - 가상 AAR 일지 A·B [교육용가상자료]**

| 시각 | AAR 일지 A (불필요한 중지가 잦은 사례) | AAR 일지 B (중지가 늦은 사례) |
| --- | --- | --- |
| 09:00 | UV-2 투입, 링크 3등급 이하면 중지하는 규칙 | UD-1 투입, 배터리 20% 이하면 복귀하는 규칙 |
| 09:40 | 링크 3등급 순간 하락 4회, 모두 중지 | 배터리 표시 35%에서 급감 시작 |
| 10:10 | 하루 6건 중 4건이 불필요한 중지로 기록 | 배터리 표시 22%, 복귀 판단 대기 |
| 10:30 | 보급 도착이 반나절 지연 | 표시 18%에서 복귀 지시, 도착 전 강제 착륙 |
| 11:00 | 현장에서 임계값을 무시하자는 의견 | 원격 회수 요청까지 25분 소요 |

**가. 제공 자료 2 - 가상 운용 20회 기록의 배터리 중지 임계값 3안 비교 [교육용가상자료]**

| 안 | 복귀 임계 잔량 | 불필요한 중지 건수(20회 중) | 중지가 늦어 장비를 잃은 건수(20회 중) |
| --- | --- | --- | --- |
| 안 1 | 30% | 7 | 0 |
| 안 2 | 25% | 3 | 1 |
| 안 3 | 20% | 1 | 3 |

**나. 작성 서식 1 - 계획과 실제 차이, 원인 분류(사람·절차·정보·환경, 개인 비난 표현 금지)**

| 번호 | 계획 | 실제 | 원인 분류 | 원인 설명(사실 중심) | 개선안 |
| --- | --- | --- | --- | --- | --- |
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |
| 4 |  |  |  |  |  |
| 5 |  |  |  |  |  |

**나. 작성 서식 2 - 레드팀 쟁점·대응(5건 이상)**

| 번호 | 기준서의 취약한 전제 | 레드팀 비판 | 작성 팀 반박 또는 수정안 | 경고 문구 |
| --- | --- | --- | --- | --- |
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |
| 5 |  |  |  |  |
| 6 |  |  |  |  |

**나. 작성 서식 3 - 반대 방향 위험 점검**

| 위험 | 설명 | 대응 |
| --- | --- | --- |
| 임계값을 너무 높여 보급 지연 |  |  |
| 임계값을 너무 낮춰 장비 손실 |  |  |
| 현장에서 중지 조건을 무시하는 관행 |  |  |

**나. 작성 서식 4 - 수정 반영 계획**

| 번호 | 수정할 문서와 위치 | 수정 내용 | 책임 직책(가상) | 기한 |
| --- | --- | --- | --- | --- |
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |

**나. 작성 서식 5 - 중지 조건 민감도 재설계(임계값 3안 비교)**

| 안 | 얻는 점 | 잃는 점 | 채택 여부 | 채택 이유 또는 보완책(예: 임계값 이중화) |
| --- | --- | --- | --- | --- |
| 안 1 |  |  |  |  |
| 안 2 |  |  |  |  |
| 안 3 |  |  |  |  |

**다. 점검표**

| 점검 항목 | 확인 |
| --- | --- |
| 일지 A·B의 계획과 실제 차이를 5건 이상 적었다 | (   ) |
| 원인을 사람·절차·정보·환경으로 분류하고 개인 비난 표현이 없다 | (   ) |
| 레드팀 쟁점 5건 이상에 반박 또는 수정안을 100% 적었다 | (   ) |
| 취약한 전제에 경고 문구를 명시했다 | (   ) |
| 임계값 3안을 비교하고 선택 근거를 적었다 | (   ) |

## 단계별 수행 안내와 미션 전용 AI 활용

### 활동 1. 계획·실제·집계 분모를 구분한다

**무엇을 하나요?** 1. 단계 개요 목표: 가상 AAR(사후검토) 일지에 기록된 데이터와 수치를 분석할 때, 주관적 해석이나 단순 합산으로 인한 오류를 방지하기 위해 계획(Plan), 실제 수행(Actual), 그리고 집계 분모(Denominator)의 개념을 명확히 분리하고 산식에 대입하여 직접 계산합니다. 수행 방식: AAR A와 B의 5건 사이의 차이점과 3안의 20회 집계 결과를 직접 수치화하고 대조하여, 데이터 왜곡 없이 객관적인 실패 원인 분석의 기초를 다집니다. 2. 세부 수행 절차 계획(Plan)과 실제(Actual) 데이터 분리 정의: 무인 보급 임무 투입 전 수립했던 최초 계획 지표와, 실제 현장 또는 시나리오 수행 과정에서 측정된 실제 실행 값을 각각 추출하여 대조 기준을 세웁니다. 집계 분모(Denominator)의 고정 및 확인: 비율이나 평균을 산출할 때 기준이 되는 전체 집계 분모(예: 전체 관제 횟수, 총 테스트 건수 등)를 명확히 고정하여, 표본 변동에 따른 통계적 오류를 예방합니다. AAR A·B 간의 5건 차이 및 3안 20회 집계 결과 직접 계산: AAR 사례 A와 B에서 나타난 5가지 주요 편차 항목을 계산하고, 3안 루브릭에 제시된 20회의 반복 집계 결과를 수식에 맞춰 직접 도출합니다. 3. 주의 및 검증 사항 수치 계산 과정에서 단순한 계획상 목표치와 실제 현장 데이터(또는 모의 로그)를 혼동하여 산출하지 않도록 주의해야 합니다. 본 단계에서 도출된 계산 결과와 집계 분모 데이터는 후속 미션(미션 8의 2～4단계 및 미션 9)의 핵심 근거 자료로 활용되므로, 작성 내역과 산식을 문서에 명확히 기록해야 합니다.

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 2. AI 레드팀과 전제를 검토한다

**무엇을 하나요?** 1. 단계 개요 목표: 가상 AAR(사후검토) 일지에서 도출된 기존 무인 보급장비 운용 및 중지 조건의 핵심 전제들을 AI 레드팀 관점 에서 비판적으로 검토받고, 취약한 전제 5건 이상을 도출하여 비교안의 조건, 가치 기준, 그리고 미제공 원시 자료의 한계를 철저히 확인합니다. 수행 방식: 확증 편향과 자의적 해석을 배제하기 위해 AI를 적대적 검증자(레드팀)로 활용하여, 무인 보급장비 실패 사례 속 숨겨진 가정과 논리적 허점을 다면적으로 해체하고 검증합니다. 2. 세부 수행 절차 AI 레드팀 비판 프롬프트 적용 및 취약 전제 5건 이상 도출: 기존 중지 조건과 책임 한계의 정당성에 대해 AI에게 비판적 검토를 요구하고, 논리적 허점이나 비약이 담긴 취약한 전제를 최소 5가지 이상 식별합니다. 비교안의 조건 및 가치 기준 대조: 대안으로 제시된 시나리오별 비교안이 어떤 전제 조건과 가치 기준(예: 안전성 우선 vs 임무 완수 우선)을 바탕으로 설정되었는지 정밀하게 비교합니다. 미제공 원시 자료 및 정보 공백 식별: 현재 공유된 가상 자료만으로는 확인할 수 없는 미제공 원시 자료(예: 실시간 통신 대역폭 상세 로그, 장비 센서 오작동 원시 데이터 등)의 영역을 명확히 구분하고, 정보 공백으로 인한 판단의 한계를 기록합니다. 3. 주의 및 검증 사항 AI 레드팀의 비판 내용을 무조건 수용하거나 형식적으로 넘기지 않고, 원본 자료 및 가상 AAR 일지의 실제 내용과 교차 검증해야 합니다. 본 단계에서 발굴된 5건 이상의 취약 전제와 비교안 검토 결과는 후속 미션(3단계 반박·수정 및 4단계 최종 보고서)의 핵심 근거가 되므로 상세히 문서화해야 합니다.

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 3. 모든 비판에 근거로 대응한다

**무엇을 하나요?** 1. 단계 개요 목표: 앞선 2단계에서 AI 레드팀이 제기한 다양한 비판과 지적 사항에 대해, 사람·절차·정보·환경의 4대 영역별 가설 과 원본 데이터 및 AAR 일지 를 철저히 상호 대조하여 각 비판에 대한 타당성을 검증하고, 반박(Defend), 수정(Revise), 또는 경고(Warn) 조치 를 최종 결정합니다. 수행 방식: 비판을 무조건 수용하거나 자의적으로 기각하지 않고, 오직 객관적 원자료와 가설 간의 정합성을 근거로 삼아 판단하며, 본 단계에서는 새로운 대안이나 추가 시나리오를 임의로 창출하지 않고 기존 문제점의 진단과 교정에 집중합니다. 2. 세부 수행 절차 4대 영역(사람·절차·정보·환경)별 가설과 원자료 대조: AI 레드팀의 비판 내용이 인적 과실, 운용 절차의 결함, 정보(데이터)의 누락/오류, 환경적 제약 중 어느 영역에 해당하는지 분류하고, 실제 AAR 원자료의 기록과 대조합니다. 비판별 대응 방향 결정 (반박·수정·경고): 반박(Defend): 레드팀의 지적이 원자료의 명확한 사실과 배치되거나 과도한 가정에 기반한 경우, 객관적 증거를 들어 논리적으로 방어합니다. 수정(Revise): 지적된 내용이 타당하여 기존 중지 조건이나 체크리스트에 구조적 허점이 있다고 판단될 경우, 구체적인 개선 방향을 도출합니다. 경고(Warn): 제도적 수정까지는 아니더라도 현장 운용 시 치명적인 리스크로 작용할 수 있는 요소에 대해 주의보를 발령합니다. 신규 결과 생성 통제: 본 단계는 기존 비판을 검증하고 교정하는 과정이므로, 범위를 벗어난 새로운 안이나 결과물을 임의로 확장하여 만들지 않습니다. 3. 주의 및 검증 사항 추상적이거나 주관적인 느낌에 의존하지 않고, 원자료의 구체적인 수치와

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 4. 조건부 비교와 수정 계획을 넘긴다

**무엇을 하나요?** 1. 단계 개요 목표: 앞선 1～3단계를 통해 도출된 3안의 장단점, 핵심 선택 기준, 반대 위험 요소, 그리고 구체적인 반영 책임과 기한 을 명확히 정돈하여 후속 미션인 미션 9(피드백 반영 및 실무 적용성 검증)와 미션 10(표준 체크리스트 완성)으로 정확하고 누락 없이 인계합니다. 수행 방식: 단편적인 분석 결과로 끝나지 않도록, 검증된 수정 계획과 조건부 비교 안을 체계적으로 이관하여 현장 적용성과 책임 소재를 명확히 합니다. 2. 세부 수행 절차 3안 비교 결과의 장단점 및 선택 기준 정돈: 미션 8 과정에서 검토된 3가지 대안(3안)의 구조적 장점과 한계점(단점)을 최종 정리하고, 어떤 기준(안전성, 신속성, 경제성 등)에 근거하여 해당 안이 선택되었는지 명시합니다. 반대 위험(Risk) 및 예외 상황 식별: 해당 수정안을 현장에 도입할 때 발생할 수 있는 잠재적 반대 의견, 역효과, 혹은 기술적·제도적 위험 요소를 상세히 기록합니다. 반영 책임(Owner) 및 추진 기한 설정: 도출된 수정 계획과 체크리스트 보완 사항을 실제로 이행하고 반영해야 할 주체(책임자/부서)와 구체적인 이행 기한을 확정합니다. 미션 9 및 미션 10과의 연계 이관: 정리된 데이터와 최종 검토 보고서 내용을 후속 미션들의 입력 자료로 정확히 넘길 수 있도록 문서 체계를 완비합니다. 3. 주의 및 검증 사항 책임 소재와 기한이 모호하게 작성되면 후속 미션(미션 9의 실무 검증 및 미션 10의 표준 지침서 완성)에서 실행력이 떨어지므로, 담당자와 일정을 구체적으로 명시해야 합니다. 본 단계의 산출물은 최종 결과물인 reports/log05_redteam_critique_report.md 및 관련 양식 파일에 빠짐없

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

## 제출 전에 확인할 기준

**평가 항목 1:** [정량적 평가] [선택] 실패 원인 규명·민감도 재설계 — 도출된 비판적 쟁점 5건 이상에 대해 반박 및 수정 대안을 100% 수립했는가, 임계값 3안의 민감도를 비교해 선택 근거를 적었는가, 취약한 전제(Assumptions)에 경고 문구를 명시했는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영. PASS 증거 가상 AAR 일지에서 무인 보급 실패 사례의 원인을 사람·절차·정보·환경으로 규명하고, 중지 조건이 너무 민감하거나 둔감한 지점을 찾아 재설계한다.의판단이제공자료·역할책임·교육용가정과연결됩니다. 사실/제안/미확인과실제/모의수행을구분합니다. 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다. 미실시를 PASS로 쓰지 않는다. 파일: reports/log05_redteam_critique_report.md, LOG05_M08_forms.md, forms/LOG05_M08_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다. 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다. 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**중요:** 정답·실제 측정값·사람의 서명이나 승인을 AI가 만들어 주지 않도록 합니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m08", "steps": [{"title": "계획·실제·집계 분모를 구분한다"}, {"title": "AI 레드팀과 전제를 검토한다"}, {"title": "모든 비판에 근거로 대응한다"}, {"title": "조건부 비교와 수정 계획을 넘긴다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("ipywidgets가 필요합니다. 폐쇄망에서는 사전 설치된 로컬 Jupyter 또는 제공된 Markdown 양식을 사용하세요.") from exc

MISSION_DATA = json.loads(MISSION_JSON)
CODE = MISSION_DATA['code']
OUT_DIR = Path(CODE + '_results')
OUT_DIR.mkdir(exist_ok=True)
BACKUP = OUT_DIR / (CODE + '_record.json')
fields = {}
for no, item in enumerate(MISSION_DATA['steps'], 1):
    fields['activity_%02d' % no] = widgets.Textarea(
        value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
        description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
        style={'description_width':'75px'})
for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                   ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                   ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
    fields[key] = widgets.Textarea(value='', description=label,
       layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

def payload():
    return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'answers':{key:w.value for key,w in fields.items()},
            'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

def save_record(_=None):
    obj=payload()
    if BACKUP.exists():
        stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
    BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
    status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

def export_json(_=None):
    save_record()
    raw=BACKUP.read_bytes()
    try:
        from google.colab import files
        files.download(str(BACKUP))
    except ImportError:
        print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

def restore_record(_=None):
    try:
        obj=json.loads(restore_input.value)
        if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
        save_record()  # 복원 전 백업 저장
        for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
        status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
    except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

def export_review_zip(_=None):
    save_record()
    target=OUT_DIR / (CODE+'_review.zip')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
        zz.write(BACKUP,arcname=BACKUP.name)
        text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
        zz.writestr(CODE+'_work.md',text)
        zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
    try:
        from google.colab import files
        files.download(str(target))
    except ImportError:
        print('Colab이 아닙니다. ZIP 위치:',target.resolve())

for i,item in enumerate(MISSION_DATA['steps'],1):
    display(Markdown('### 활동 %d. %s' % (i,item['title'])))
    display(fields['activity_%02d' % i])
for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
 ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
    display(Markdown('### '+label));display(fields[key])
buttons=[]
for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
    b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)


## 마무리

작성 JSON과 검토용 ZIP을 내려받아 보관하세요. ZIP은 **검토용**이며 평가 PASS나 교관 승인 증명이 아닙니다. 전 단계 결과가 필요하면 본인의 실제 이전 기록을 사용하고, 기록이 없으면 **미실시**로 남기세요.